# CCLM + CIRA Colab Notebook

Run everything directly in notebook cells (no clone/install cells).

Goal: compare CIRA method accuracy vs baseline accuracy.

## Step 1: Paths and dataset detection

In [31]:
from pathlib import Path
import sys

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'outputs' / 'run_colab_notebook'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SRC = ROOT / 'src'
if SRC.exists() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Optional local defaults if files already exist in uploaded folder structure.
DATASET_PATH = ROOT / 'data' / 'cira_lab_dataset.json'
if not DATASET_PATH.exists():
    DATASET_PATH = None

CONFIG_PATH = ROOT / 'configs' / 'base.yaml'
if not CONFIG_PATH.exists():
    CONFIG_PATH = None

print('ROOT:', ROOT)
print('DATASET:', DATASET_PATH if DATASET_PATH is not None else 'upload in next cell')
print('CONFIG:', CONFIG_PATH if CONFIG_PATH is not None else 'upload in next cell')
print('OUTPUT_DIR:', OUTPUT_DIR)

ROOT: /content
DATASET: upload in next cell
CONFIG: upload in next cell
OUTPUT_DIR: /content/outputs/run_colab_notebook


In [33]:
from google.colab import files

# Upload required files directly from Colab UI.
# Expected names: cira_lab_dataset.json and optionally base.yaml
uploaded = files.upload()

print('Uploaded files:', list(uploaded.keys()))

UPLOAD_DATASET_PATH = None
UPLOAD_CONFIG_PATH = None

if 'cira_lab_dataset.json' in uploaded:
    UPLOAD_DATASET_PATH = OUTPUT_DIR / 'uploaded_cira_lab_dataset.json'
    UPLOAD_DATASET_PATH.write_bytes(uploaded['cira_lab_dataset.json'])
    print('Saved dataset to:', UPLOAD_DATASET_PATH)

if 'base.yaml' in uploaded:
    UPLOAD_CONFIG_PATH = OUTPUT_DIR / 'uploaded_base.yaml'
    UPLOAD_CONFIG_PATH.write_bytes(uploaded['base.yaml'])
    print('Saved config to:', UPLOAD_CONFIG_PATH)

Saving cira_lab_dataset.json to cira_lab_dataset.json
Uploaded files: ['cira_lab_dataset.json']
Saved dataset to: /content/outputs/run_colab_notebook/uploaded_cira_lab_dataset.json


## Step 1.5: Dataset in notebook (for Colab single-file run)
This cell keeps the dataset inside the ipynb so no external JSON file is required.

## Step 2: Imports

In [34]:
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset

try:
    from google.colab import files
    print('Using Google Colab runtime imports with inline fallback implementations.')
    raise ModuleNotFoundError('Use inline notebook implementations')
except ModuleNotFoundError:
    print('Using inline notebook implementations.')

    TOKEN_PATTERN = re.compile(r"[A-Za-z0-9_]+")

    @dataclass
    class Sample:
        id: int
        space: str
        initial: str
        new: str
        distractor: str
        query: str
        answer: str

    class Vocab:
        PAD = '<pad>'
        UNK = '<unk>'

        def __init__(self) -> None:
            self.stoi: dict[str, int] = {self.PAD: 0, self.UNK: 1}
            self.itos: list[str] = [self.PAD, self.UNK]

        def add_token(self, token: str) -> None:
            if token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

        def add_text(self, text: str) -> None:
            for tok in tokenize(text):
                self.add_token(tok)

        def encode(self, text: str) -> list[int]:
            return [self.stoi.get(tok, self.stoi[self.UNK]) for tok in tokenize(text)]

        @property
        def size(self) -> int:
            return len(self.itos)

    class CIRALabDataset(Dataset):
        def __init__(self, samples: list[Sample], vocab: Vocab, answer_to_id: dict[str, int]) -> None:
            self.samples = samples
            self.vocab = vocab
            self.answer_to_id = answer_to_id

        def __len__(self) -> int:
            return len(self.samples)

        def __getitem__(self, idx: int) -> dict[str, Any]:
            s = self.samples[idx]
            return {
                'id': s.id,
                'space': s.space,
                'query_ids': self.vocab.encode(s.query),
                'wm_ids': self.vocab.encode(s.new),
                'lm_initial_ids': self.vocab.encode(s.initial),
                'lm_distractor_ids': self.vocab.encode(s.distractor),
                'label': self.answer_to_id[s.answer],
                'query': s.query,
                'answer': s.answer,
                'initial': s.initial,
                'distractor': s.distractor,
            }

    def tokenize(text: str) -> list[str]:
        return [t.lower() for t in TOKEN_PATTERN.findall(text)]

    def read_samples(path: str | Path) -> list[Sample]:
        with open(path, 'r', encoding='utf-8') as f:
            raw = json.load(f)
        return [Sample(**item) for item in raw]

    def build_vocab(samples: list[Sample]) -> Vocab:
        vocab = Vocab()
        for s in samples:
            vocab.add_text(s.initial)
            vocab.add_text(s.new)
            vocab.add_text(s.distractor)
            vocab.add_text(s.query)
        return vocab

    def build_answer_space(samples: list[Sample]) -> tuple[dict[str, int], dict[int, str]]:
        answers = sorted({s.answer for s in samples})
        a2i = {a: i for i, a in enumerate(answers)}
        i2a = {i: a for a, i in a2i.items()}
        return a2i, i2a

    def split_samples(
        samples: list[Sample],
        val_ratio: float,
        test_ratio: float,
        seed: int,
    ) -> tuple[list[Sample], list[Sample], list[Sample]]:
        assert val_ratio >= 0 and test_ratio >= 0 and (val_ratio + test_ratio) < 1.0
        rng = np.random.default_rng(seed)
        idx = np.arange(len(samples))
        rng.shuffle(idx)

        test_size = int(round(len(samples) * test_ratio))
        val_size = int(round(len(samples) * val_ratio))

        test_idx = idx[:test_size]
        val_idx = idx[test_size : test_size + val_size]
        train_idx = idx[test_size + val_size :]

        to_list = lambda indices: [samples[int(i)] for i in indices]
        return to_list(train_idx), to_list(val_idx), to_list(test_idx)

    def _pad(sequences: list[list[int]], pad_id: int = 0) -> tuple[torch.Tensor, torch.Tensor]:
        max_len = max(len(s) for s in sequences)
        ids = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)
        mask = torch.zeros((len(sequences), max_len), dtype=torch.bool)
        for i, seq in enumerate(sequences):
            ids[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
            mask[i, : len(seq)] = True
        return ids, mask

    def collate_fn(batch: list[dict[str, Any]]) -> dict[str, Any]:
        query_ids, query_mask = _pad([x['query_ids'] for x in batch])
        wm_ids, wm_mask = _pad([x['wm_ids'] for x in batch])
        lm_i_ids, lm_i_mask = _pad([x['lm_initial_ids'] for x in batch])
        lm_d_ids, lm_d_mask = _pad([x['lm_distractor_ids'] for x in batch])
        labels = torch.tensor([x['label'] for x in batch], dtype=torch.long)

        return {
            'id': [x['id'] for x in batch],
            'space': [x['space'] for x in batch],
            'query_ids': query_ids,
            'query_mask': query_mask,
            'wm_ids': wm_ids,
            'wm_mask': wm_mask,
            'lm_initial_ids': lm_i_ids,
            'lm_initial_mask': lm_i_mask,
            'lm_distractor_ids': lm_d_ids,
            'lm_distractor_mask': lm_d_mask,
            'labels': labels,
            'answer': [x['answer'] for x in batch],
            'initial': [x['initial'] for x in batch],
            'distractor': [x['distractor'] for x in batch],
        }

    class TextEncoder(nn.Module):
        def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, dropout: float) -> None:
            super().__init__()
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
            self.dropout = nn.Dropout(dropout)

        def forward(self, token_ids: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
            emb = self.dropout(self.embedding(token_ids))
            output, _ = self.rnn(emb)
            masked = output * mask.unsqueeze(-1)
            pooled = masked.sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp_min(1)
            return pooled

    @dataclass
    class CIRAOutput:
        logits: torch.Tensor
        sim_wm: torch.Tensor
        sim_lm: torch.Tensor
        sim_dist: torch.Tensor
        interference: torch.Tensor
        gate: torch.Tensor

    class CIRAClassifier(nn.Module):
        def __init__(
            self,
            vocab_size: int,
            num_labels: int,
            embedding_dim: int,
            hidden_dim: int,
            dropout: float,
            confidence_wm_prior: float,
            confidence_lm_prior: float,
        ) -> None:
            super().__init__()
            self.encoder = TextEncoder(vocab_size, embedding_dim, hidden_dim, dropout)
            repr_dim = hidden_dim * 2

            self.interference_scorer = nn.Sequential(
                nn.Linear(repr_dim * 4, repr_dim),
                nn.ReLU(),
                nn.Linear(repr_dim, 1),
            )

            self.gate_proj = nn.Linear(4, 1)
            self.classifier = nn.Sequential(
                nn.Linear(repr_dim * 4, repr_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(repr_dim, num_labels),
            )

            self.confidence_wm_prior = confidence_wm_prior
            self.confidence_lm_prior = confidence_lm_prior

        def forward(
            self,
            query_ids: torch.Tensor,
            query_mask: torch.Tensor,
            wm_ids: torch.Tensor,
            wm_mask: torch.Tensor,
            lm_initial_ids: torch.Tensor,
            lm_initial_mask: torch.Tensor,
            lm_distractor_ids: torch.Tensor,
            lm_distractor_mask: torch.Tensor,
        ) -> CIRAOutput:
            q = self.encoder(query_ids, query_mask)
            wm = self.encoder(wm_ids, wm_mask)
            lm_i = self.encoder(lm_initial_ids, lm_initial_mask)
            lm_d = self.encoder(lm_distractor_ids, lm_distractor_mask)

            sim_wm = F.cosine_similarity(q, wm, dim=-1)
            sim_i = F.cosine_similarity(q, lm_i, dim=-1)
            sim_d = F.cosine_similarity(q, lm_d, dim=-1)

            lm_weights = torch.softmax(torch.stack([sim_i, sim_d], dim=1), dim=1)
            lm = lm_weights[:, 0:1] * lm_i + lm_weights[:, 1:2] * lm_d
            sim_lm = F.cosine_similarity(q, lm, dim=-1)

            inter_features = torch.cat([wm, lm, torch.abs(wm - lm), q], dim=-1)
            interference = torch.sigmoid(self.interference_scorer(inter_features)).squeeze(-1)

            wm_conf = torch.sigmoid(sim_wm) * self.confidence_wm_prior
            lm_conf = torch.sigmoid(sim_lm) * self.confidence_lm_prior * (1.0 - lm_weights[:, 1])

            gate_features = torch.stack([wm_conf, lm_conf, interference, sim_wm - sim_d], dim=-1)
            gate = torch.sigmoid(self.gate_proj(gate_features)).squeeze(-1)

            refined_context = gate.unsqueeze(-1) * wm + (1.0 - gate.unsqueeze(-1)) * lm
            fused = torch.cat([q, refined_context, q * refined_context, torch.abs(q - refined_context)], dim=-1)
            logits = self.classifier(fused)

            return CIRAOutput(logits=logits, sim_wm=sim_wm, sim_lm=sim_lm, sim_dist=sim_d, interference=interference, gate=gate)

    @dataclass
    class EvalResult:
        loss: float
        accuracy: float
        stale_hits: int
        distractor_hits: int

    def _accuracy(preds: list[int], labels: list[int]) -> float:
        if not labels:
            return 0.0
        hits = sum(int(p == y) for p, y in zip(preds, labels))
        return hits / len(labels)

    def _stale_or_distractor_hits(
        pred_answers: list[str],
        initial_texts: list[str],
        distractor_texts: list[str],
    ) -> tuple[int, int]:
        stale_hits = 0
        distractor_hits = 0
        for ans, initial, distractor in zip(pred_answers, initial_texts, distractor_texts):
            a = ans.lower()
            if a in initial.lower():
                stale_hits += 1
            if a in distractor.lower():
                distractor_hits += 1
        return stale_hits, distractor_hits

    def evaluate_model(
        model: CIRAClassifier,
        data_loader: DataLoader,
        id_to_answer: dict[int, str],
        device: torch.device,
    ) -> EvalResult:
        model.eval()
        total_loss = 0.0
        criterion = torch.nn.CrossEntropyLoss()

        all_preds: list[int] = []
        all_labels: list[int] = []
        pred_answers: list[str] = []
        all_initial: list[str] = []
        all_distractor: list[str] = []

        with torch.no_grad():
            for batch in data_loader:
                batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

                out = model(
                    query_ids=batch['query_ids'],
                    query_mask=batch['query_mask'],
                    wm_ids=batch['wm_ids'],
                    wm_mask=batch['wm_mask'],
                    lm_initial_ids=batch['lm_initial_ids'],
                    lm_initial_mask=batch['lm_initial_mask'],
                    lm_distractor_ids=batch['lm_distractor_ids'],
                    lm_distractor_mask=batch['lm_distractor_mask'],
                )
                loss = criterion(out.logits, batch['labels'])
                total_loss += loss.item() * batch['labels'].size(0)

                pred = torch.argmax(out.logits, dim=-1)
                pred_list = pred.detach().cpu().tolist()
                all_preds.extend(pred_list)
                all_labels.extend(batch['labels'].detach().cpu().tolist())
                pred_answers.extend([id_to_answer[i] for i in pred_list])
                all_initial.extend(batch['initial'])
                all_distractor.extend(batch['distractor'])

        stale_hits, distractor_hits = _stale_or_distractor_hits(pred_answers, all_initial, all_distractor)
        acc = _accuracy(all_preds, all_labels)
        denom = max(1, len(all_labels))
        return EvalResult(loss=total_loss / denom, accuracy=acc, stale_hits=stale_hits, distractor_hits=distractor_hits)

    def set_seed(seed: int) -> None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    def load_yaml(path: str | Path) -> dict[str, Any]:
        import yaml
        with open(path, 'r', encoding='utf-8') as f:
            return yaml.safe_load(f)

    def device_for_training() -> torch.device:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

Using Google Colab runtime imports with inline fallback implementations.
Using inline notebook implementations.
Torch: 2.10.0+cpu
CUDA available: False


## Step 3: Build data loaders

In [35]:
active_config_path = globals().get('UPLOAD_CONFIG_PATH', None) or CONFIG_PATH
cfg = load_yaml(active_config_path) if active_config_path is not None else {
    'seed': 42,
    'train': {
        'val_ratio': 0.2,
        'test_ratio': 0.2,
        'batch_size': 16,
        'lr': 1e-3,
        'weight_decay': 1e-4,
        'epochs': 30,
        'gradient_clip': 1.0,
    },
    'model': {
        'embedding_dim': 128,
        'hidden_dim': 128,
        'dropout': 0.2,
        'confidence_wm_prior': 0.7,
        'confidence_lm_prior': 0.3,
    },
}
set_seed(cfg['seed'])
device = device_for_training()

active_dataset_path = globals().get('UPLOAD_DATASET_PATH', None) or DATASET_PATH
if active_dataset_path is not None and Path(active_dataset_path).exists():
    samples = read_samples(active_dataset_path)
    print('Using dataset:', active_dataset_path)
else:
    raise FileNotFoundError(
        'Dataset not found. Upload cira_lab_dataset.json in the upload cell before running this step.'
    )

train_s, val_s, test_s = split_samples(
    samples,
    val_ratio=cfg['train']['val_ratio'],
    test_ratio=cfg['train']['test_ratio'],
    seed=cfg['seed']
)

vocab = build_vocab(train_s + val_s + test_s)
answer_to_id, id_to_answer = build_answer_space(train_s + val_s + test_s)

train_ds = CIRALabDataset(train_s, vocab, answer_to_id)
val_ds = CIRALabDataset(val_s, vocab, answer_to_id)
test_ds = CIRALabDataset(test_s, vocab, answer_to_id)

train_loader = DataLoader(train_ds, batch_size=cfg['train']['batch_size'], shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=cfg['train']['batch_size'], shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg['train']['batch_size'], shuffle=False, collate_fn=collate_fn)

print({'train': len(train_ds), 'val': len(val_ds), 'test': len(test_ds)})

Using dataset: /content/outputs/run_colab_notebook/uploaded_cira_lab_dataset.json
{'train': 6, 'val': 2, 'test': 2}


## Step 4: Train CIRA method

In [36]:
set_seed(cfg['seed'])
model = CIRAClassifier(
    vocab_size=vocab.size,
    num_labels=len(answer_to_id),
    embedding_dim=cfg['model']['embedding_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    dropout=cfg['model']['dropout'],
    confidence_wm_prior=cfg['model']['confidence_wm_prior'],
    confidence_lm_prior=cfg['model']['confidence_lm_prior'],
).to(device)

optimizer = AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
criterion = torch.nn.CrossEntropyLoss()

history = []
best_val_acc = -1.0
best_path = OUTPUT_DIR / 'best_model.pt'

for epoch in range(1, cfg['train']['epochs'] + 1):
    model.train()
    run_loss = 0.0
    run_n = 0
    for batch in train_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = model(
            query_ids=batch['query_ids'],
            query_mask=batch['query_mask'],
            wm_ids=batch['wm_ids'],
            wm_mask=batch['wm_mask'],
            lm_initial_ids=batch['lm_initial_ids'],
            lm_initial_mask=batch['lm_initial_mask'],
            lm_distractor_ids=batch['lm_distractor_ids'],
            lm_distractor_mask=batch['lm_distractor_mask'],
        )
        cls_loss = criterion(out.logits, batch['labels'])
        margin = torch.relu(out.sim_dist - out.sim_wm + 0.1).mean()
        loss = cls_loss + 0.25 * margin

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['train']['gradient_clip'])
        optimizer.step()

        bs = batch['labels'].size(0)
        run_loss += loss.item() * bs
        run_n += bs

    tr_loss = run_loss / max(1, run_n)
    val = evaluate_model(model, val_loader, id_to_answer, device)
    history.append({'epoch': float(epoch), 'train_loss': tr_loss, 'val_loss': val.loss, 'val_accuracy': val.accuracy})

    if val.accuracy > best_val_acc:
        best_val_acc = val.accuracy
        torch.save({'model_state_dict': model.state_dict(), 'vocab': vocab.itos, 'answer_to_id': answer_to_id, 'config': cfg}, best_path)

    if epoch % 10 == 0 or epoch == 1:
        print(f"epoch={epoch:03d} train_loss={tr_loss:.4f} val_loss={val.loss:.4f} val_acc={val.accuracy:.3f}")

(OUTPUT_DIR / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best validation accuracy:', round(best_val_acc, 4))

epoch=001 train_loss=2.3601 val_loss=2.3003 val_acc=0.000
epoch=010 train_loss=0.5719 val_loss=3.0778 val_acc=0.000
epoch=020 train_loss=0.0070 val_loss=4.5993 val_acc=0.000
epoch=030 train_loss=0.0002 val_loss=5.7541 val_acc=0.000
Best validation accuracy: 0.0


## Step 5: Test CIRA accuracy

In [37]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
cira_test = evaluate_model(model, test_loader, id_to_answer, device)

cira_metrics = {
    'test_loss': cira_test.loss,
    'test_accuracy': cira_test.accuracy,
    'stale_hits': cira_test.stale_hits,
    'distractor_hits': cira_test.distractor_hits
}

(OUTPUT_DIR / 'metrics_cira.json').write_text(json.dumps(cira_metrics, indent=2), encoding='utf-8')
print(json.dumps(cira_metrics, indent=2))

{
  "test_loss": 2.348496675491333,
  "test_accuracy": 0.0,
  "stale_hits": 0,
  "distractor_hits": 0
}


## Step 6: Train baseline (no interference-margin term)

In [38]:
set_seed(cfg['seed'])
base_model = CIRAClassifier(
    vocab_size=vocab.size,
    num_labels=len(answer_to_id),
    embedding_dim=cfg['model']['embedding_dim'],
    hidden_dim=cfg['model']['hidden_dim'],
    dropout=cfg['model']['dropout'],
    confidence_wm_prior=cfg['model']['confidence_wm_prior'],
    confidence_lm_prior=cfg['model']['confidence_lm_prior'],
).to(device)

base_optim = AdamW(base_model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
base_criterion = torch.nn.CrossEntropyLoss()

best_base_val = -1.0
best_base_state = None

for epoch in range(1, cfg['train']['epochs'] + 1):
    base_model.train()
    for batch in train_loader:
        batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        out = base_model(
            query_ids=batch['query_ids'],
            query_mask=batch['query_mask'],
            wm_ids=batch['wm_ids'],
            wm_mask=batch['wm_mask'],
            lm_initial_ids=batch['lm_initial_ids'],
            lm_initial_mask=batch['lm_initial_mask'],
            lm_distractor_ids=batch['lm_distractor_ids'],
            lm_distractor_mask=batch['lm_distractor_mask'],
        )
        loss = base_criterion(out.logits, batch['labels'])

        base_optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(base_model.parameters(), cfg['train']['gradient_clip'])
        base_optim.step()

    base_val = evaluate_model(base_model, val_loader, id_to_answer, device)
    if base_val.accuracy > best_base_val:
        best_base_val = base_val.accuracy
        best_base_state = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}

base_model.load_state_dict(best_base_state)
base_test = evaluate_model(base_model, test_loader, id_to_answer, device)

baseline_metrics = {
    'test_loss': base_test.loss,
    'test_accuracy': base_test.accuracy,
    'stale_hits': base_test.stale_hits,
    'distractor_hits': base_test.distractor_hits
}

(OUTPUT_DIR / 'metrics_baseline.json').write_text(json.dumps(baseline_metrics, indent=2), encoding='utf-8')
print(json.dumps(baseline_metrics, indent=2))

{
  "test_loss": 2.3487472534179688,
  "test_accuracy": 0.0,
  "stale_hits": 0,
  "distractor_hits": 0
}


## Step 7: Accuracy comparison (your method vs baseline)

In [39]:
gain = cira_metrics['test_accuracy'] - baseline_metrics['test_accuracy']
comparison = {
    'cira_accuracy': cira_metrics['test_accuracy'],
    'baseline_accuracy': baseline_metrics['test_accuracy'],
    'absolute_gain': gain,
    'claim_supported': bool(gain > 0)
}

(OUTPUT_DIR / 'accuracy_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
print(json.dumps(comparison, indent=2))

if gain > 0:
    print('Conclusion: Your CIRA method performs better on this split.')
elif gain == 0:
    print('Conclusion: Tie on this split.')
else:
    print('Conclusion: Baseline is better on this split.')

{
  "cira_accuracy": 0.0,
  "baseline_accuracy": 0.0,
  "absolute_gain": 0.0,
  "claim_supported": false
}
Conclusion: Tie on this split.
